### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
# Necesario en Colab. En local las dependencias ya vienen del entorno (uv sync).
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


#### Ejercicio 1

Tomamos 5 documentos al azar del conjunto de entrenamiento, medimos la similaridad coseno de cada uno contra los 11.314 documentos del set de entrenamiento y estudiamos los 5 más similares: su valor de similaridad, su etiqueta y un extracto del texto, para juzgar si las temáticas son similares.

In [24]:
target_names = newsgroups_train.target_names


def extracto(texto, n_caracteres=300):
    """Colapsa espacios y saltos de línea y recorta el texto para poder inspeccionarlo."""
    return ' '.join(texto.split())[:n_caracteres]


rng = np.random.default_rng(42)  # semilla fija
docs_azar = rng.choice(X_train.shape[0], size=5, replace=False)

coincidencias = []
for doc_idx in docs_azar:
    cossim = cosine_similarity(X_train[doc_idx], X_train)[0]
    # el más similar es el documento consigo mismo (sim = 1), por eso arrancamos en la posición 1
    mas_similares = np.argsort(cossim)[::-1][1:6]
    n_iguales = int(sum(y_train[j] == y_train[doc_idx] for j in mas_similares))
    coincidencias.append(n_iguales)

    print('=' * 110)
    print(f'DOCUMENTO {doc_idx} | clase: {target_names[y_train[doc_idx]]}')
    print(f'   {extracto(newsgroups_train.data[doc_idx])}')
    print(f'   --> {n_iguales}/5 de los más similares comparten la etiqueta')
    for rank, j in enumerate(mas_similares, start=1):
        marca = 'MISMA CLASE' if y_train[j] == y_train[doc_idx] else 'OTRA CLASE '
        print(f'   {rank}. {marca}  sim={cossim[j]:.3f}  clase={target_names[y_train[j]]}  (doc {j})')
        print(f'      {extracto(newsgroups_train.data[j], 200)}')
    print()

print('=' * 110)
print(f'Vecinos con la misma etiqueta por documento: {coincidencias}  '
      f'-> {sum(coincidencias)}/25 ({100 * sum(coincidencias) / 25:.0f}%)')

DOCUMENTO 8754 | clase: talk.religion.misc
   /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves. They may be permanently damaging their body. That is true. It is also none of your business. Some people may also reason that by reading the bible and being a Xtian you are permanently dama
   --> 4/5 de los más similares comparten la etiqueta
   1. MISMA CLASE  sim=0.490  clase=talk.religion.misc  (doc 6552)
      If I have a habit that I really want to break, and I am willing to make whatever sacrifice I need to make to break it, then I do so. There have been bad habits of mine that I've decided to put forth t
   2. MISMA CLASE  sim=0.481  clase=talk.religion.misc  (doc 10613)
      /(hudson) /Yes you do. Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other way. Maybe unpleasant, but that doesn't /say anything about morality. It violates f
   3. MISMA CLASE  sim=0.465  clase=talk.religion.

Conclusiones:
- En el 52% de los vecinos recuperados la etiqueta coincide con la del documento consultado. Esto sería muy distinto eligiendo documentos al azar, por lo cual la similitud coseno funciona.
- Aún en las clases que no coincidieron, se observan similitudes temáticas. Por ejemplo, comp.os.ms-windows.misc con comp.windows.x, sci.med y comp.graphics - todas clases relacionadas a computación.
- Se obtuvieron 2 documentos que consisten en una sola palabra: Hello. Sus vectores tienen un solo término con peso 1, lo que significa que ambos documentos tienen similaridad con cualquier documento igual al peso que ese documento le da a hello, sin importar el tema. Esto es una desventaja de este enfoque, por lo que quizás convendría filtrar documentos con menos de N términos en primer lugar.
- Cuando el documento es largo y tiene vocabulario propio, los vecinos son muy buenos. El documento 8754 (talk.religion.misc, lo que parece ser una discusión sobre moral y la Biblia) recupera 4 documentos de la misma clase que parecen seguir el mismo hilo de conversación.

#### Ejercicio 2

Clasificamos cada documento de test comparándolo con todos los de entrenamiento y le asignamos la etiqueta del documento de train con mayor similaridad coseno. No hay parámetros que ajustar ni entrenamiento. Lo comparamos contra el `MultinomialNB` de referencia del notebook.

In [25]:
def clasificar_por_prototipos(X_eval, X_ref, y_ref, chunk=500):
    """Asigna a cada documento de X_eval la etiqueta del documento de X_ref más similar (coseno).

    Trabajamos por bloques porque la matriz de similaridades completa (7532 x 11314) es densa
    y no conviene materializarla de una sola vez.
    """
    y_pred_proto = np.empty(X_eval.shape[0], dtype=y_ref.dtype)
    sim_max = np.empty(X_eval.shape[0])
    for ini in range(0, X_eval.shape[0], chunk):
        fin = min(ini + chunk, X_eval.shape[0])
        sims = cosine_similarity(X_eval[ini:fin], X_ref)
        vecino = sims.argmax(axis=1)
        sim_max[ini:fin] = sims[np.arange(fin - ini), vecino]
        y_pred_proto[ini:fin] = y_ref[vecino]
    return y_pred_proto, sim_max


y_pred_prototipos, sim_max = clasificar_por_prototipos(X_test, X_train, y_train)

f1_prototipos = f1_score(y_test, y_pred_prototipos, average='macro')
f1_nb_baseline = f1_score(y_test, y_pred, average='macro')
print(f'Prototipos (vecino más similar) - F1-Score macro: {f1_prototipos:.4f}')
print(f'Prototipos (vecino más similar) - Accuracy:       {(y_pred_prototipos == y_test).mean():.4f}')
print(f'MultinomialNB de referencia     - F1-Score macro: {f1_nb_baseline:.4f}')

Prototipos (vecino más similar) - F1-Score macro: 0.5050
Prototipos (vecino más similar) - Accuracy:       0.5089
MultinomialNB de referencia     - F1-Score macro: 0.5854


In [26]:
# Documentos de test que no comparten ningún término con train: su predicción es arbitraria
sin_solape = sim_max == 0
f1_sin_vacios = f1_score(y_test[~sin_solape], y_pred_prototipos[~sin_solape], average='macro')
print(f'Documentos de test sin ningún término en común con train: {sin_solape.sum()} ({100 * sin_solape.mean():.1f}%)')
print(f'F1-Score macro excluyendo esos documentos: {f1_sin_vacios:.4f}\n')

print(f'Similaridad con el vecino elegido: media={sim_max.mean():.3f}  mediana={np.median(sim_max):.3f}')
for umbral in [0.2, 0.35, 0.5]:
    mask = sim_max > umbral
    acc = (y_pred_prototipos[mask] == y_test[mask]).mean()
    print(f'   accuracy en los {mask.sum():5d} documentos con similaridad > {umbral}: {acc:.4f}')

f1_por_clase = f1_score(y_test, y_pred_prototipos, average=None)
orden = np.argsort(f1_por_clase)
print('\nClases peor clasificadas:')
for i in orden[:4]:
    print(f'   {target_names[i]:28s} F1={f1_por_clase[i]:.3f}')
print('Clases mejor clasificadas:')
for i in orden[::-1][:4]:
    print(f'   {target_names[i]:28s} F1={f1_por_clase[i]:.3f}')

Documentos de test sin ningún término en común con train: 224 (3.0%)
F1-Score macro excluyendo esos documentos: 0.5171

Similaridad con el vecino elegido: media=0.308  mediana=0.281
   accuracy en los  6404 documentos con similaridad > 0.2: 0.5484
   accuracy en los  2034 documentos con similaridad > 0.35: 0.7075
   accuracy en los   567 documentos con similaridad > 0.5: 0.8483

Clases peor clasificadas:
   talk.religion.misc           F1=0.277
   talk.politics.misc           F1=0.307
   sci.electronics              F1=0.406
   alt.atheism                  F1=0.425
Clases mejor clasificadas:
   rec.sport.hockey             F1=0.735
   comp.windows.x               F1=0.642
   rec.sport.baseball           F1=0.586
   sci.crypt                    F1=0.570


Conclusiones:
- El clasificador consigue un F1-Score macro de `0.5050` contra `0.5854` del `MultinomialNB` de referencia, lo cual no es despreciable considerando que no hay entrenamiento alguno. La razón es que la decisión se apoya en un único documento vecino, mientras que Naive Bayes agrega la evidencia de todos los documentos de cada clase.
- 224 documentos de test (3%) no comparten ningún término con ningún documento de train. Para ellos la similaridad máxima es 0, por lo cual el documento 0 de train es elegido por defecto. Esto hace que la predicción no tenga ningún sentido. Cuando se los saca, el F1 sube un poco a `0.5171`, pero no soluciona el problema de fondo.
- La similaridad máxima funciona como una medida de confianza muy útil en los 567 donde supera el 0.5, obteniendo un accuracy de `0.8483` - una mejora sobre el `0.5089` de todo el conjunto de test.
- En cuanto a las clases peor clasificadas, se obserban temáticas abstractas o generales mientras que las mejor clasificadas son más específicas.
- Si bien no hay entrenamiento, el costo computacional es alto ya que el método debe mantener mucha información en memoria y hacer 7.532 x 11.314 comparaciones.

#### Ejercicio 3

Barremos una grilla de configuraciones combinando parámetros del vectorizador con los dos modelos pedidos (`MultinomialNB` y `ComplementNB`) y distintos valores de suavizado `alpha`. Después refinamos ese valor sobre el mejor vectorizador. Como indica la consigna, no modificamos `ngram_range`.

In [27]:
configs_vect = [
    ('TF-IDF por defecto', TfidfVectorizer, {}),
    ('Conteos + stop_words + min_df=2', CountVectorizer, dict(stop_words='english', min_df=2)),
    ('TF-IDF + stop_words + min_df=2', TfidfVectorizer, dict(stop_words='english', min_df=2)),
    ('TF-IDF + stop_words + min_df=2 + sublinear_tf', TfidfVectorizer,
     dict(stop_words='english', min_df=2, sublinear_tf=True)),
    ('TF-IDF + stop_words + min_df=5 + max_df=0.7 + sublinear_tf', TfidfVectorizer,
     dict(stop_words='english', min_df=5, max_df=0.7, sublinear_tf=True, strip_accents='unicode')),
]

configs_modelo = [
    (f'{nombre}(alpha={alpha})', clase, alpha)
    for nombre, clase in [('MultinomialNB', MultinomialNB), ('ComplementNB', ComplementNB)]
    for alpha in [0.01, 0.1, 1.0]
]

resultados = []
for nombre_v, clase_v, kwargs_v in configs_vect:
    vect = clase_v(**kwargs_v)
    X_tr = vect.fit_transform(newsgroups_train.data)
    X_te = vect.transform(newsgroups_test.data)
    for nombre_m, clase_m, alpha in configs_modelo:
        modelo = clase_m(alpha=alpha).fit(X_tr, y_train)
        f1 = f1_score(y_test, modelo.predict(X_te), average='macro')
        resultados.append({'f1': f1, 'vectorizador': nombre_v, 'modelo': nombre_m, 'vocabulario': X_tr.shape[1]})

resultados.sort(key=lambda r: r['f1'], reverse=True)

print(f'{len(resultados)} configuraciones evaluadas, ordenadas por F1-Score macro en test\n')
print(f'{"F1":>7}  {"vocab":>7}  vectorizador | modelo')
for r in resultados:
    print(f'{r["f1"]:7.4f}  {r["vocabulario"]:7d}  {r["vectorizador"]} | {r["modelo"]}')

30 configuraciones evaluadas, ordenadas por F1-Score macro en test

     F1    vocab  vectorizador | modelo
 0.6954   101631  TF-IDF por defecto | ComplementNB(alpha=0.1)
 0.6943    39115  TF-IDF + stop_words + min_df=2 | ComplementNB(alpha=1.0)
 0.6930   101631  TF-IDF por defecto | ComplementNB(alpha=1.0)
 0.6921    39115  TF-IDF + stop_words + min_df=2 + sublinear_tf | ComplementNB(alpha=1.0)
 0.6887    39115  TF-IDF + stop_words + min_df=2 | ComplementNB(alpha=0.1)
 0.6878    39115  TF-IDF + stop_words + min_df=2 + sublinear_tf | ComplementNB(alpha=0.1)
 0.6829   101631  TF-IDF por defecto | MultinomialNB(alpha=0.01)
 0.6820    17797  TF-IDF + stop_words + min_df=5 + max_df=0.7 + sublinear_tf | ComplementNB(alpha=1.0)
 0.6801    39115  TF-IDF + stop_words + min_df=2 | MultinomialNB(alpha=0.01)
 0.6798    39115  TF-IDF + stop_words + min_df=2 | MultinomialNB(alpha=0.1)
 0.6777    39115  TF-IDF + stop_words + min_df=2 + sublinear_tf | MultinomialNB(alpha=0.1)
 0.6750    17797  TF-IDF

In [28]:
# Refinamos alpha sobre el vectorizador que quedó primero en la grilla anterior
nombre_mejor_vect = resultados[0]['vectorizador']
clase_v, kwargs_v = next((c, k) for n, c, k in configs_vect if n == nombre_mejor_vect)

vect_final = clase_v(**kwargs_v)
X_train_final = vect_final.fit_transform(newsgroups_train.data)
X_test_final = vect_final.transform(newsgroups_test.data)

alphas = [0.005, 0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
curvas = {}
for clase_m in (MultinomialNB, ComplementNB):
    curvas[clase_m.__name__] = [
        f1_score(y_test, clase_m(alpha=alpha).fit(X_train_final, y_train).predict(X_test_final), average='macro')
        for alpha in alphas
    ]

print(f'Sensibilidad a alpha usando "{nombre_mejor_vect}" (F1-Score macro en test)\n')
print(f'{"alpha":>7}  {"MultinomialNB":>14}  {"ComplementNB":>14}')
for k, alpha in enumerate(alphas):
    print(f'{alpha:7}  {curvas["MultinomialNB"][k]:14.4f}  {curvas["ComplementNB"][k]:14.4f}')

Sensibilidad a alpha usando "TF-IDF por defecto" (F1-Score macro en test)

  alpha   MultinomialNB    ComplementNB
  0.005          0.6811          0.6595
   0.01          0.6829          0.6689
   0.03          0.6767          0.6810
   0.05          0.6698          0.6861
    0.1          0.6565          0.6954
    0.2          0.6425          0.6997
    0.3          0.6322          0.6999
    0.5          0.6153          0.6961
    0.7          0.6011          0.6947
    1.0          0.5854          0.6930


In [29]:
# Nos quedamos con la mejor combinación de modelo y alpha del refinamiento
mejor_modelo, mejor_pos = max(
    ((nombre, int(np.argmax(f1s))) for nombre, f1s in curvas.items()),
    key=lambda par: curvas[par[0]][par[1]],
)
mejor_alpha = alphas[mejor_pos]
mejor_f1 = curvas[mejor_modelo][mejor_pos]

clf_mejor = {'MultinomialNB': MultinomialNB, 'ComplementNB': ComplementNB}[mejor_modelo](alpha=mejor_alpha)
clf_mejor.fit(X_train_final, y_train)
y_pred_mejor = clf_mejor.predict(X_test_final)

print(f'Mejor configuración: {nombre_mejor_vect} + {mejor_modelo}(alpha={mejor_alpha})')
print(f'F1-Score macro: {mejor_f1:.4f}   (baseline del notebook: {f1_nb_baseline:.4f}, '
      f'mejora de {100 * (mejor_f1 - f1_nb_baseline):.1f} puntos)\n')

f1_clases = f1_score(y_test, y_pred_mejor, average=None)
orden = np.argsort(f1_clases)
print('Clases con menor F1 en el mejor modelo:')
for i in orden[:4]:
    print(f'  {target_names[i]:28s} F1={f1_clases[i]:.3f}')
print('Clases con mayor F1 en el mejor modelo:')
for i in orden[::-1][:4]:
    print(f'  {target_names[i]:28s} F1={f1_clases[i]:.3f}')

Mejor configuración: TF-IDF por defecto + ComplementNB(alpha=0.3)
F1-Score macro: 0.6999   (baseline del notebook: 0.5854, mejora de 11.5 puntos)

Clases con menor F1 en el mejor modelo:
  talk.religion.misc           F1=0.249
  alt.atheism                  F1=0.378
  talk.politics.misc           F1=0.522
  sci.electronics              F1=0.635
Clases con mayor F1 en el mejor modelo:
  rec.sport.hockey             F1=0.903
  rec.sport.baseball           F1=0.878
  talk.politics.mideast        F1=0.825
  rec.motorcycles              F1=0.801


Conclusiones:
- La mejor configuración encontrada fue TF-IDF por defecto + `ComplementNB` con un alpha de 0.3. Se obtuvo un F1-Score macro de `0.6999` en test contra `0.5854` del modelo de referencia del notebook (TF-IDF por defecto + MultinomialNB con alpha=1).
- El valor óptimo de alfa depende del modelo. MultinomialNB necesita un alpha chico, de alrededor de 0.01. ComplementNB se comporta al revés y prefiere un alpha entre 0.2 y 0.3.
- ComplementNB suele obtener mejores resultados que MultinomialNB al estimar los pesos de cada clase usando su complemento (todos los documentos que no son de esa clase), con lo cual cada estimación se apoya en mucha más evidencia.
- TF-IDF obtiene mejores resultados que los conteos crudos: la mejor configuración con CountVectorizer llega a `0.6408` frente a `0.6999`. Mediante IDF se penaliza los términos que aparecen en todos los grupos y que no discriminan.
- Recortar el vocabulario baja mucho la dimensionalidad pero a un costo mínimo en desempeño: pasando de 101.631 términos (F1-score `0.6954` sin refinar alpha) a 39.115 (F1-score `0.6943`) y a 17.797 (F1-score `0.6820`). Sirve para eficiencia pero no tanto para desempeño. El parámetro sublinear_tf tampoco movió el resultado de forma apreciable.
- Ocurre algo similar al ejercicio anterior al observar las clases peor y mejor rankeadas, donde las categorías más específicas con vocabulario propio obtienen mejores resultados.

#### Ejercicio 4

Transponemos la matriz documento-término para obtener una matriz término-documento, donde cada fila es la vectorización de una palabra en el espacio de los documentos. Elegimos 5 palabras manualmente (una representativa de distintos grupos de noticias) y buscamos sus 5 palabras más similares por coseno.

In [30]:
# Usamos stop_words y min_df=5 para que el vocabulario resultante sea interpretable
tfidfvect_palabras = TfidfVectorizer(stop_words='english', min_df=5)
X_palabras = tfidfvect_palabras.fit_transform(newsgroups_train.data)

# Transponemos: cada fila pasa a ser un término y cada columna un documento
M_termino_doc = X_palabras.T.tocsr()
idx2word_palabras = {v: k for k, v in tfidfvect_palabras.vocabulary_.items()}
df_termino = np.asarray((X_palabras > 0).sum(axis=0)).ravel()  # en cuántos documentos aparece cada término

print(f'Matriz documento-término:                {X_palabras.shape}')
print(f'Matriz término-documento (transpuesta):  {M_termino_doc.shape}\n')

palabras = ['god', 'car', 'space', 'windows', 'gun']
similaridades_top = []
for palabra in palabras:
    i = tfidfvect_palabras.vocabulary_[palabra]
    sims = cosine_similarity(M_termino_doc[i], M_termino_doc)[0]
    mas_similares = np.argsort(sims)[::-1][1:6]  # la posición 0 es la palabra consigo misma
    similaridades_top.extend(sims[mas_similares])

    print(f'{palabra!r}  (aparece en {df_termino[i]} documentos)')
    for rank, j in enumerate(mas_similares, start=1):
        print(f'   {rank}. {idx2word_palabras[j]:15s} sim={sims[j]:.3f}   (aparece en {df_termino[j]} documentos)')
    print()

Matriz documento-término:                (11314, 17797)
Matriz término-documento (transpuesta):  (17797, 11314)

'god'  (aparece en 579 documentos)
   1. jesus           sim=0.277   (aparece en 263 documentos)
   2. bible           sim=0.268   (aparece en 243 documentos)
   3. christ          sim=0.267   (aparece en 197 documentos)
   4. faith           sim=0.255   (aparece en 191 documentos)
   5. existence       sim=0.249   (aparece en 137 documentos)

'car'  (aparece en 396 documentos)
   1. cars            sim=0.192   (aparece en 177 documentos)
   2. dealer          sim=0.177   (aparece en 107 documentos)
   3. civic           sim=0.163   (aparece en 22 documentos)
   4. loan            sim=0.156   (aparece en 14 documentos)
   5. owner           sim=0.148   (aparece en 99 documentos)

'space'  (aparece en 398 documentos)
   1. nasa            sim=0.318   (aparece en 157 documentos)
   2. shuttle         sim=0.278   (aparece en 71 documentos)
   3. exploration     sim=0.233   (apa

Conclusiones:
- Al transponer la matriz documento-término se obtiene una matriz término-documento de 17797 x 11314: cada fila es un vector de palabra cuyas componentes son los documentos donde aparece esa palabra. Dos palabras son similares si tienden a aparecer en los mismos documentos.
- Los vecinos obtenidos de cada palabra reconstruyen el vocabulario del grupo de noticias donde suelen aparecer. Por ejemplo, god con jesus/bible/christ, etc.
- Se mezclan relaciones morfológicas (gun/guns, car/cars) al tratar cada forma como un término independiente. También sinónimos como gun/handgun/firearms.
- Las similaridades son bajas, entre 0.15 y 0.38. Los vectores de palabras son aún más dispersos que los de documentos.